# Student Performance — Data Selection, Manipulation & Visualization

This notebook explores the student performance dataset (`Expanded_data_with_more_features.csv` or the uploaded CSV file). It includes data inspection, cleaning, column selection, summary statistics, and visualizations of math, reading, and writing scores.

**Before running:** place the CSV file in the same folder as this notebook. If your file has a different name, update `DATA_PATH` in the next cell.

## 1. Import libraries

We use **Pandas** for data manipulation, **Matplotlib** and **Seaborn** for charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

print("Libraries imported successfully.")

## 2. Load the dataset

The code tries several common filenames. If none is found, change `DATA_PATH` to your CSV filename.

In [ ]:
possible_files = [
    "Expanded_data_with_more_features.csv",
    "b119bb30-d943-4808-ab85-f269b231ebfb.csv",
    "StudentsPerformance.csv"
]
DATA_PATH = next((name for name in possible_files if Path(name).exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "CSV file not found. Put the dataset beside this notebook and set DATA_PATH "
        "to its exact filename."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

## 3. Explore the data

Inspect column names, data types, duplicate rows, and missing values before analysis.

In [ ]:
print("Column names:")
print(df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nDuplicate rows:", df.duplicated().sum())

missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary.to_frame("Missing values"))

## 4. Data selection and manipulation

- Remove automatically generated index columns such as `Unnamed: 0`.
- Standardize column names by removing surrounding spaces.
- Identify the score columns and relevant demographic/study columns.
- Create `AverageScore` from math, reading, and writing scores.

In [ ]:
# Remove unwanted index columns if present
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.match(r"^Unnamed(: ?\d+)?$")]

score_cols = [col for col in ["MathScore", "ReadingScore", "WritingScore"] if col in df.columns]
if len(score_cols) < 3:
    raise ValueError(f"Expected score columns were not all found. Available columns: {df.columns.tolist()}")

# Keep a focused selection for the analysis
preferred_cols = [
    "Gender", "EthnicGroup", "ParentEduc", "LunchType", "TestPrep",
    "PracticeSport", "NrSiblings", "WklyStudyHours", *score_cols
]
selected_cols = [col for col in preferred_cols if col in df.columns]
analysis_df = df[selected_cols].copy()

# Convert score columns to numeric in case they were imported as text
for col in score_cols:
    analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

# Create a combined performance measure
analysis_df["AverageScore"] = analysis_df[score_cols].mean(axis=1)

print("Original dataset shape:", df.shape)
print("Selected dataset shape:", analysis_df.shape)
display(analysis_df.head())

## 5. Summary statistics

Review the distribution of scores and compare average scores across the three subjects.

In [ ]:
display(analysis_df[score_cols + ["AverageScore"]].describe().round(2))

mean_scores = analysis_df[score_cols].mean().sort_values(ascending=False)
display(mean_scores.rename("Mean score").to_frame())

## 6. Missing values visualization

This chart shows how many values are missing in each selected column. It is useful for deciding what cleaning may be needed.

In [ ]:
missing_counts = analysis_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if missing_counts.empty:
    print("The selected columns have no missing values.")
else:
    sns.barplot(x=missing_counts.values, y=missing_counts.index)
    plt.title("Missing Values by Column")
    plt.xlabel("Number of missing values")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()

## 7. Score distributions

Histograms help us see where student scores are concentrated and whether the distributions are skewed.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, score_cols):
    sns.histplot(data=analysis_df, x=col, kde=True, bins=20, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel("Score")
plt.tight_layout()
plt.show()

## 8. Compare average scores by gender

The grouped bar chart compares the mean math, reading, and writing scores for each gender category available in the dataset.

In [ ]:
if "Gender" in analysis_df.columns:
    gender_means = analysis_df.groupby("Gender", dropna=False)[score_cols].mean().reset_index()
    gender_long = gender_means.melt(id_vars="Gender", var_name="Subject", value_name="Average score")
    sns.barplot(data=gender_long, x="Gender", y="Average score", hue="Subject")
    plt.title("Average Scores by Gender")
    plt.xlabel("Gender")
    plt.ylabel("Average score")
    plt.legend(title="Subject")
    plt.tight_layout()
    plt.show()
else:
    print("Gender column not available; chart skipped.")

## 9. Study hours and performance

This chart compares average scores by weekly study-hours category. The categories are sorted in their existing order where possible.

In [ ]:
if "WklyStudyHours" in analysis_df.columns:
    study_order = analysis_df["WklyStudyHours"].dropna().astype(str).unique().tolist()
    study_means = analysis_df.groupby("WklyStudyHours", dropna=False)[score_cols].mean().reset_index()
    study_long = study_means.melt(id_vars="WklyStudyHours", var_name="Subject", value_name="Average score")
    study_long["WklyStudyHours"] = study_long["WklyStudyHours"].astype(str)
    sns.barplot(data=study_long, x="WklyStudyHours", y="Average score", hue="Subject", order=study_order)
    plt.title("Average Scores by Weekly Study Hours")
    plt.xlabel("Weekly study hours")
    plt.ylabel("Average score")
    plt.xticks(rotation=20, ha="right")
    plt.legend(title="Subject")
    plt.tight_layout()
    plt.show()
else:
    print("WklyStudyHours column not available; chart skipped.")

## 10. Test preparation and scores

This comparison helps examine whether average scores differ between students in the available test-preparation categories. It shows association, not proof of causation.

In [ ]:
if "TestPrep" in analysis_df.columns:
    prep_means = analysis_df.groupby("TestPrep", dropna=False)[score_cols].mean().reset_index()
    prep_long = prep_means.melt(id_vars="TestPrep", var_name="Subject", value_name="Average score")
    sns.barplot(data=prep_long, x="TestPrep", y="Average score", hue="Subject")
    plt.title("Average Scores by Test Preparation")
    plt.xlabel("Test preparation category")
    plt.ylabel("Average score")
    plt.legend(title="Subject")
    plt.tight_layout()
    plt.show()
else:
    print("TestPrep column not available; chart skipped.")

## 11. Score relationships

A correlation heatmap shows how strongly the numerical score columns move together. Values near +1 indicate a strong positive linear relationship.

In [ ]:
corr = analysis_df[score_cols + ["AverageScore"]].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Between Student Scores")
plt.tight_layout()
plt.show()

## 12. Average score by lunch type

A boxplot compares the spread of the combined average score between lunch categories, including the median and potential outliers.

In [ ]:
if "LunchType" in analysis_df.columns:
    sns.boxplot(data=analysis_df, x="LunchType", y="AverageScore")
    plt.title("Average Score Distribution by Lunch Type")
    plt.xlabel("Lunch type")
    plt.ylabel("Average score")
    plt.tight_layout()
    plt.show()
else:
    print("LunchType column not available; chart skipped.")

## 13. Key findings (generate after running)

Use the output from the charts and summary statistics to write conclusions. The cell below prints simple data-driven observations based on the dataset you loaded.

In [ ]:
print("DATASET SUMMARY")
print("- Number of students:", f"{len(analysis_df):,}")
print("- Number of selected columns:", len(analysis_df.columns))
print("- Mean scores:")
for subject, value in analysis_df[score_cols].mean().sort_values(ascending=False).items():
    print(f"  • {subject}: {value:.2f}")
print(f"- Overall average score: {analysis_df['AverageScore'].mean():.2f}")
print("- Missing values in selected data:", int(analysis_df.isna().sum().sum()))
print("\nRemember: group differences and correlations show associations; they do not establish causation.")

## Conclusion

This notebook demonstrated data loading, inspection, selection, manipulation, and visualization using Pandas, Matplotlib, and Seaborn. The charts explore score distributions and relationships with student characteristics. Final conclusions should be based on the outputs after running all cells with the provided dataset.